In [2]:
"""
T_n_full_scan.py

Полный перебор всех пар n-значных чисел (A, B), A < B.

Перекрёстная разность:
  Δ_n(A, B) = Σ_{0 ≤ i < j ≤ n−1} (a_i·b_j − a_j·b_i)

где a_i, b_i — цифры A и B (a_0 = единицы).

T_n(K) — число пар с A < B и Δ_n = K.

Проверяет гипотезу:
  T_n(K) ≠ T_n(−K) для всех K ≠ 0,
  где хотя бы одно из значений положительно.

Без экспорта таблиц. Только заключение и сводка.

Запуск:
  python3 T_n_full_scan.py        # n = 3 по умолчанию
  python3 T_n_full_scan.py 4      # если хочется n = 4 (долго)
"""

from collections import defaultdict
import sys


def digits_of(num, n):
    """Список цифр числа num длины n, младшая цифра первая."""
    return [num // (10 ** i) % 10 for i in range(n)]


def b_coefficients(b_digits):
    """Коэффициенты c_i такие, что Δ = Σ a_i · c_i."""
    n = len(b_digits)
    coeffs = []
    prefix = 0
    suffix = sum(b_digits)
    for i in range(n):
        suffix -= b_digits[i]
        coeffs.append(suffix - prefix)
        prefix += b_digits[i]
    return coeffs


def main(n_digits=3):
    N_MIN = 10 ** (n_digits - 1)
    N_MAX = 10 ** n_digits - 1
    COUNT = N_MAX - N_MIN + 1
    total_pairs = COUNT * (COUNT - 1) // 2

    print("=" * 60)
    print(f"ПОЛНЫЙ ПЕРЕБОР {n_digits}-ЗНАЧНЫХ ЧИСЕЛ")
    print("=" * 60)
    print(f"Чисел:                     {COUNT}")
    print(f"Пар с A < B (итераций):    {total_pairs}")
    if COUNT > 2000:
        print(f"ВНИМАНИЕ: это может занять минуты и больше.")
    print()

    # Предвычисление
    a_digits_list = []
    b_coeffs_list = []
    for num in range(N_MIN, N_MAX + 1):
        d = digits_of(num, n_digits)
        a_digits_list.append(d)
        b_coeffs_list.append(b_coefficients(d))

    T = defaultdict(int)
    iterations = 0

    # Основной цикл
    for i in range(COUNT):
        a = a_digits_list[i]
        for j in range(i + 1, COUNT):
            c = b_coeffs_list[j]
            k = 0
            for t in range(n_digits):
                k += a[t] * c[t]
            T[k] += 1
            iterations += 1

    # --- Проверки ---
    sum_T = sum(T.values())
    k_min = min(T.keys()) if T else 0
    k_max = max(T.keys()) if T else 0
    distinct = len(T)
    theoretical_max = 81 * n_digits * (n_digits - 1) // 2

    print(f"Итераций выполнено:        {iterations}")
    print(f"Совпадает с ожиданием:     {iterations == total_pairs}")
    print(f"Сумма T(K):                {sum_T}")
    print(f"Совпадает с ожиданием:     {sum_T == total_pairs}")
    print()
    print(f"Диапазон K:                [{k_min}, {k_max}]")
    print(f"Достижимых значений K:     {distinct}")
    print(f"Теоретическая граница:     [{-theoretical_max}, {theoretical_max}]")
    print()

    # --- Гипотеза ---
    all_K = set()
    for k in T:
        all_K.add(k)
        all_K.add(-k)
    all_K.discard(0)

    violations = []
    for k in all_K:
        if T.get(k, 0) == T.get(-k, 0):
            violations.append(k)
    violations_abs = sorted(set(abs(k) for k in violations))

    print("=" * 60)
    print("ЗАКЛЮЧЕНИЕ")
    print("=" * 60)
    if not violations_abs:
        print(f"Гипотеза ПОДТВЕРЖДЕНА для n = {n_digits}.")
        print(f"T(K) ≠ T(−K) для всех K ≠ 0,")
        print(f"где хотя бы одно из значений положительно.")
    else:
        print(f"Гипотеза ОПРОВЕРГНУТА для n = {n_digits}.")
        print(f"Контрпримеры: |K| ∈ {violations_abs}")
        for k in violations_abs:
            print(f"  K = {k}:  T({k}) = {T.get(k, 0)},  "
                  f"T({-k}) = {T.get(-k, 0)}")
    print()

    # --- Топ значений ---
    top = sorted(T.items(), key=lambda p: (-p[1], p[0]))[:5]
    print(f"Топ-5 значений T(K):")
    for k, v in top:
        print(f"  K = {k}:  T = {v}")
    print()


if __name__ == "__main__":
    n = 3
    if len(sys.argv) > 1:
        try:
            n = int(sys.argv[1])
        except ValueError:
            n = 3
    main(n)

ПОЛНЫЙ ПЕРЕБОР 3-ЗНАЧНЫХ ЧИСЕЛ
Чисел:                     900
Пар с A < B (итераций):    404550

Итераций выполнено:        404550
Совпадает с ожиданием:     True
Сумма T(K):                404550
Совпадает с ожиданием:     True

Диапазон K:                [-162, 234]
Достижимых значений K:     373
Теоретическая граница:     [-243, 243]

ЗАКЛЮЧЕНИЕ
Гипотеза ПОДТВЕРЖДЕНА для n = 3.
T(K) ≠ T(−K) для всех K ≠ 0,
где хотя бы одно из значений положительно.

Топ-5 значений T(K):
  K = 0:  T = 7573
  K = 24:  T = 5438
  K = 36:  T = 5117
  K = 12:  T = 5060
  K = 30:  T = 5057

